In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import re
import plotly.graph_objects as go

from scipy.stats import gaussian_kde
import numpy as np



### SECTION 1: DATA PREP
df = pd.read_csv('data/all_devices_summary_7.csv')
df = df.rename(columns={'Amplifier_board': 'Amplifier_Board'})

def extract_wafer_code_flexible(wafer_origin: str) -> str:
    if pd.isna(wafer_origin) or not isinstance(wafer_origin, str):
        return "Unknown"
    wafer_origin = wafer_origin.strip().upper()
    patterns = [r'^([A-Z]\d+)', r'^([A-Z]\d+)[\\-_ ]', r'([A-Z]\d+)[\\-_ ]?\d']
    for pattern in patterns:
        match = re.search(pattern, wafer_origin)
        if match:
            return match.group(1)
    return "Unknown"

df['Wafer_Type'] = df['Wafer_origin'].astype(str).apply(extract_wafer_code_flexible)
df['yield_numeric'] = df['Yield'].astype(str).str.rstrip("%").replace("", "0").astype(float)
df = df[df['yield_numeric'] >= 60]  # Keep your yield threshold

valid_wafers = sorted(df[df['Wafer_Type'] != 'Unknown']['Wafer_Type'].unique())
print(f"Valid wafers: {valid_wafers}")

imp_cols = [col for col in df.columns if col.startswith('Imp_Ch')]
for col in imp_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

imp_long = pd.melt(df, id_vars=['Wafer_Type', 'Device_ID', 'Amplifier_Board'], #this line of code is transforming the DataFrame from a wide format to a long format using the pd.melt function. The id_vars parameter specifies the columns that will remain unchanged (Wafer_Type, Device_ID, Amplifier_Board), while the value_vars parameter specifies the columns that will be unpivoted (the impedance columns starting with 'Imp_Ch'). The var_name parameter sets the name of the new column that will contain the names of the original columns (Channel), and the value_name parameter sets the name of the new column that will contain the values from those columns (Imp_Ohm). The dropna method is then used to remove any rows where Imp_Ohm is NaN, ensuring that only valid impedance measurements are included in the resulting DataFrame.
                   value_vars=imp_cols, var_name='Channel', value_name='Imp_Ohm').dropna(subset=['Imp_Ohm'])


imp_long['Above_1M'] = imp_long['Imp_Ohm'] >= 1e6  # Keep threshold logic



##END OF SECTION 1: DATA PREP







## SECTION 2: L1 DEVICES WITH ANY CHANNEL < 5kΩ
# Focus only on L1 wafer
l1_data = imp_long[imp_long["Wafer_Type"] == "L1"].copy()

# Devices that have at least one sensor/channel < 5 kΩ
threshold = 5_000  # 5 kΩ
l1_bad_devices = (
    l1_data[l1_data["Imp_Ohm"] < threshold]
    .groupby("Device_ID")
    .size()
    .reset_index(name="n_channels_below_5k")
)

print(f"Number of L1 devices with any channel < 5kΩ: {len(l1_bad_devices)}")
print(l1_bad_devices.sort_values("Device_ID").head(20))  # preview

### End of SECTION 2:L1 DEVICES WITH ANY CHANNEL < 5kΩ






# SECTION 3:WAFER COLOR MAPPING. Use to identify electrode types by wafer. Adjust colors as needed for your specific wafer types and preferences.
wafer_colors = {
    'L1': "green", 
    'R12': 'blue',
    'R3': "red",# if i set both r3 and r4 to orange, i am saying they are both the same electrode type, which is fine if that is the case. If they are different types, you may want to choose different colors to distinguish them. The choice of colors is up to you and can be based on your preferences or any existing color-coding scheme you have for the wafers.
    "R4": 'red',
    "R6": "Purple",
    'R7': 'blue',
    'R8': 'blue',
    # Add more: 'R6': 'green', 'R8': 'purple', etc.
}
default_color = 'gray'





## SECTION 4: GROUPED SCATTER PLOT BY WAFER 
fig = go.Figure()

# Assign x-positions (jittered within wafer band) and colors. 
wafer_positions = {wafer: i for i, wafer in enumerate(valid_wafers)} # what this line of code is doing is creating a dictionary called wafer_positions that maps each valid wafer type to a specific x position. The enumerate function is used to assign a unique integer index to each wafer type, which will serve as the base x position for plotting the points in the scatter plot. This allows us to group points from the same wafer together on the x-axis while still allowing for jittering to spread them out within the band for better visibility.
imp_long['x_pos'] = imp_long['Wafer_Type'].map(wafer_positions) #this will be the base x position for each wafer, which means that all points from the same wafer will be centered around the same x value. We will add jitter to this base position to spread the points within the band for better visibility. what this line of code is doing is creating a new column called 'x_pos' in the imp_long DataFrame. This column is generated by mapping the 'Wafer_Type' values to their corresponding x positions defined in the wafer_positions dictionary. Each wafer type will have a specific x position, which will be used as the base position for plotting the points in the scatter plot. The points from the same wafer will be centered around this x position, and we will add jitter to spread them out within the band for better visibility.
np.random.seed(42)  # Fixed seed: same "random" every time for reproducibility. This allows scatter traces to be more visible and consistent across runs, which is helpful for debugging and comparing results. You can change the seed value to get a different jitter pattern if desired.
imp_long['jitter'] = np.random.uniform(-0.3, 0.3, len(imp_long))  # Spread points within band. jitter is a random value between -0.3 and 0.3 that will be added to the base x position to create a spread of points around the central x value for each wafer. This helps to prevent points from overlapping and makes it easier to see the distribution of points within each wafer's band. what this line of code is doing is generating a random jitter value for each point in the imp_long DataFrame. The np.random.uniform function is used to create random values uniformly distributed between -0.3 and 0.3. This jitter will be added to the base x position of each point to spread them out within the band for better visibility in the scatter plot.
imp_long['x_jittered'] = imp_long['x_pos'] + imp_long['jitter'] # what this line of code is doing is creating a new column called 'x_jittered' in the imp_long DataFrame. This column is calculated by adding the 'jitter' values to the 'x_pos' values. The 'x_pos' values represent the base x positions for each wafer, while the 'jitter' values provide a random spread around those base positions. By adding them together, we get a new x position for each point that is jittered within the band for better visibility in the scatter plot.
imp_long['color'] = imp_long['Wafer_Type'].map(wafer_colors).fillna(default_color) # what this line of code is doing is creating a new column called 'color' in the imp_long DataFrame. This column is generated by mapping the 'Wafer_Type' values to their corresponding colors defined in the wafer_colors dictionary. If a 'Wafer_Type' does not have a specified color in the wafer_colors dictionary, it will be assigned the default color (gray in this case) using the fillna method. This 'color' column can then be used to color-code the points in the scatter plot based on their wafer type.



# Good: Imp_Ohm < 1e6 (and >0, <1e9 as before)
good_mask = (~imp_long['Above_1M']) & (imp_long['Imp_Ohm'] > 0) & (imp_long['Imp_Ohm'] < 1e9) # Your "good" criteria: below 1MΩ, above 0, and below 1e9 to exclude extreme outliers. Adjust as needed.
bad_mask = imp_long['Above_1M'] & (imp_long['Imp_Ohm'] < 1e12)  # Your outlier cap for "bad" points: above 1MΩ but below 1e12 to exclude extreme outliers. Adjust as needed.




# SECTION 5: QUICK STATS BY WAFER
wafer_stats = imp_long.groupby('Wafer_Type').agg(
    N_total=('Imp_Ohm', 'count'),
    n_good=('Imp_Ohm', lambda x: good_mask.loc[x.index].sum()),
    n_bad=('Imp_Ohm', lambda x: bad_mask.loc[x.index].sum())
).round(0).astype(int)

wafer_stats['pct_good'] = (wafer_stats['n_good'] / wafer_stats['N_total'] * 100).round(1)

print(wafer_stats)  # Check: L1: N=248, n_good=200, n_bad=48, pct_good=80.6

### End of SECTION 5: QUICK STATS BY WAFER







### SECTION 6: VIOLIN PLOT WITH KDE
fig_kde = go.Figure()

wafer_positions_kde = {wafer: i for i, wafer in enumerate(valid_wafers)}  # Like your x_pos

for wafer in valid_wafers:
    wafer_data = imp_long[imp_long['Wafer_Type'] == wafer]
    valid_imp = wafer_data['Imp_Ohm'][(wafer_data['Imp_Ohm'] > 0) & (wafer_data['Imp_Ohm'] < 1e10)].values
    
    if len(valid_imp) >= 2: # Need at least 2 points for KDE
        log_y = np.log10(valid_imp) # what this line of code is doing is applying a logarithmic transformation to the valid impedance values (valid_imp) using the base 10 logarithm. The np.log10 function is used to compute the logarithm of each impedance value in the valid_imp array. This transformation is often applied to impedance data because it can help to normalize the distribution and make it easier to visualize and analyze, especially when dealing with a wide range of impedance values that can span several orders of magnitude. By taking the logarithm, we can compress the scale and highlight differences between values that would otherwise be difficult to distinguish on a linear scale.
        kde = gaussian_kde(log_y) # what this line of code is doing is creating a Gaussian Kernel Density Estimate (KDE) object using the log-transformed impedance values (log_y). The gaussian_kde function from the scipy.stats module is used to compute the KDE, which is a non-parametric way to estimate the probability density function of a random variable. In this case, it will help to visualize the distribution of the log-transformed impedance values for each wafer. The resulting kde object can be evaluated at any point to get the estimated density at that point, which will be used later to create the violin plot.
        grid = np.linspace(log_y.min() - 0.3, log_y.max() + 0.3, 200) # what this line of code is doing is creating a grid of points (grid) that will be used to evaluate the KDE for plotting the violin plot. The np.linspace function is used to generate 200 evenly spaced points between the minimum and maximum values of the log-transformed impedance (log_y), with a small padding of 0.3 added on either side to ensure that the KDE is evaluated over a slightly wider range than the actual data. This grid will allow us to compute the density values at these points, which will then be used to create the shape of the violin plot for each wafer.
        density = np.maximum(kde(grid), 1e-8) # what this line of code is doing is evaluating the KDE at the points defined in the grid and ensuring that the density values are not too small. The kde(grid) expression computes the estimated density at each point in the grid, resulting in an array of density values. The np.maximum function is then used to set a lower bound of 1e-8 on these density values, which prevents them from being too close to zero. This is important for visualization purposes, as very small density values can lead to issues when plotting the violin plot, such as extremely narrow shapes or numerical instability. By setting a minimum threshold, we ensure that the density values are suitable for creating a visually meaningful violin plot.
        computed_width = 0.35 * density / density.max() # what this line of code is doing is calculating the width of the violin plot for each point in the grid based on the density values. The density values are normalized by dividing by the maximum density (density.max()) to scale them between 0 and 1. Then, they are multiplied by a factor of 0.35 to control the overall width of the violin plot. This computed_width array will determine how wide the violin plot is at each point along the y-axis (which represents the impedance values) when we create the polygon for the violin shape. The wider sections of the violin will correspond to higher density areas, while narrower sections will correspond to lower density areas.
        
        # Multi‑wafer: offset left/right by position
        base_x = wafer_positions_kde[wafer]
        x_left = base_x - computed_width
        x_right = base_x + computed_width[::-1]
        
        y_vals = 10 ** grid # what this line of code is doing is converting the grid points back from the log scale to the original impedance scale. Since the grid was created in log space (log_y), we need to exponentiate it using base 10 to get the corresponding impedance values in the original scale. The 10 ** grid expression computes the impedance values for each point in the grid, which will be used as the y-values for plotting the violin plot. This allows us to visualize the distribution of impedance values on a logarithmic scale while still displaying the actual impedance values on the y-axis of the plot.
        y_polygon = np.concatenate([y_vals, y_vals[::-1]]) #what this line of code is doing is creating the y-coordinates for the polygon that will form the shape of the violin plot. The y_vals array contains the impedance values corresponding to the grid points, and by concatenating it with its reverse (y_vals[::-1]), we create a closed loop of y-values that will allow us to fill the area between the left and right sides of the violin plot. The resulting y_polygon array will have the impedance values in order for the left side of the violin, followed by the impedance values in reverse order for the right side, which is necessary for creating a filled polygon shape when plotting.
        x_polygon = np.concatenate([x_left, x_right]) #what this line of code is doing is creating the x-coordinates for the polygon that will form the shape of the violin plot. The x_left array contains the x-coordinates for the left side of the violin plot, while the x_right array contains the x-coordinates for the right side (in reverse order). By concatenating these two arrays, we create a single array of x-coordinates (x_polygon) that corresponds to the y-coordinates in y_polygon. This allows us to define a closed polygon shape that can be filled when plotting the violin plot, with the left and right sides of the violin properly aligned with their corresponding impedance values on the y-axis.
        
        color = wafer_colors.get(wafer, 'rgba(100,150,200,0.4)')
        
        fig_kde.add_trace(go.Scatter(
            x=x_polygon, y=y_polygon, fill='toself',
            fillcolor=color, line=dict(color='black', width=1.5),
            showlegend=True, name=f"{wafer}_violin",
            hoverinfo='skip'
        ))


fig_kde.add_hline(1e6, line_dash="dash", line_color="red", line_width=3, annotation_text="1MΩ Threshold")

fig_kde.update_layout(
    title="All Wafers KDE Violin Plot (Gaussian KDE)",
    yaxis_title="Impedance (Ω)", yaxis_type="log",
    xaxis=dict(
        tickmode='array',
        tickvals=list(wafer_positions_kde.values()),
        ticktext=list(wafer_positions_kde.keys())
    ),
    height=700
)

fig_kde.show(renderer="browser")

### End of SECTION 6: KDE Violin Plot



## SECTION 7: BOX PLOT
fig_box_num = go.Figure()

for wafer in valid_wafers:
    wafer_data = imp_long[imp_long["Wafer_Type"] == wafer]
    base_x = wafer_positions[wafer]
    color = wafer_colors.get(wafer, "gray")

    # Box centered at base_x
    fig_box_num.add_trace(
        go.Box(
            x=[base_x] * len(wafer_data),
            y=wafer_data["Imp_Ohm"],
            name=f"{wafer} box",
            line=dict(color=color),
            fillcolor="rgba(0,0,0,0.4)",# what this line of code is doing is setting the fill color of the box plot to be fully transparent. The "rgba(0,0,0,0)" value represents a color in RGBA format, where the first three values (0, 0, 0) correspond to the red, green, and blue components of the color (in this case, black), and the last value (0) represents the alpha channel, which controls the opacity of the color. By setting the alpha value to 0, we make the fill color completely transparent, meaning that only the outline of the box will be visible in the plot. This allows us to show the box plot without any filled area, which can be useful for overlaying it with other elements like scatter points without obscuring them. Changing the alpha value to 1 would make the fill color fully opaque, while values between 0 and 1 would give varying levels of transparency.
            boxpoints=False,
            showlegend=False
        )
    )

    # Jittered scatter using x_jittered you already computed
    fig_box_num.add_trace(
        go.Scatter(
            x=wafer_data["x_jittered"],
            y=wafer_data["Imp_Ohm"],
            mode="markers",
            marker=dict(color=color, size=4, opacity=0.6),
            name=wafer,
            showlegend=True
        )
    )

fig_box_num.add_hline(1e6, line_dash="dash", line_color="red", line_width=3)

fig_box_num.update_layout(
    title="All Wafers Box + Scatter (numeric x)",
    yaxis_title="Impedance (Ω)",
    yaxis_type="log",
    xaxis=dict(
        title="Wafer (jittered positions)",
        tickmode="array",
        tickvals=list(wafer_positions.values()),
        ticktext=list(wafer_positions.keys()),
    ),
    height=700
)

fig_box_num.write_html("wafer_box_scatter_numeric.html")
fig_box_num.show(renderer="browser")

## End of SECTION 7: BOX PLOT





## SECTION 8: L1 DEVICES WITH ANY CHANNEL < 5kΩ - SCATTER
# Filter long data to *only* those devices on L1
devices_to_plot = set(l1_bad_devices["Device_ID"])
l1_plot = l1_data[l1_data["Device_ID"].isin(devices_to_plot)].copy()

# Assign each L1 device a numeric x position
device_positions = {
    dev: i for i, dev in enumerate(sorted(devices_to_plot))
}
l1_plot["x_pos"] = l1_plot["Device_ID"].map(device_positions)
np.random.seed(42)
l1_plot["jitter"] = np.random.uniform(-0.3, 0.3, len(l1_plot))
l1_plot["x_jittered"] = l1_plot["x_pos"] + l1_plot["jitter"]


fig_l1 = go.Figure()

fig_l1.add_trace(
    go.Scatter(
        x=l1_plot["x_jittered"],
        y=l1_plot["Imp_Ohm"],
        mode="markers",
        marker=dict(
            color="green",   # same as L1 color, which is PoPt No Cactus.
            size=5,
            opacity=0.8
        ),
        name="L1 devices with any channel < 5kΩ",
        hovertemplate="Device %{customdata[0]}<br>Ch %{customdata[1]}<br>Imp %{y:.0f} Ω",
        customdata=np.stack([l1_plot["Device_ID"], l1_plot["Channel"]], axis=-1),
    )
)

fig_l1.add_hline(
    threshold, # this is the 5kΩ threshold. 
    line_dash="dash",
    line_color="red",
    line_width=3,
    annotation_text="5kΩ threshold"
)

fig_l1.update_layout(
    title="L1 Devices(PoPt No Cactus) with Any Sensor < 5kΩ",
    yaxis_title="Impedance (Ω)",
    yaxis_type="log",
    xaxis=dict(
        title="Device (jittered positions)",
        tickmode="array",
        tickvals=list(device_positions.values()),
        ticktext=list(device_positions.keys()),
    ),
    height=700,
)

fig_l1.write_html("L1_low_impedance_devices_scatter.html")
fig_l1.show(renderer="browser")

## End of SECTION 8: L1 DEVICES WITH ANY CHANNEL < 5kΩ - SCATTER




## SECTION 4 CONTINUED: Grouped Scatter Plot by Wafer
# Color-code by wafer (use wafer-specific colors for legend/markers)
for wafer in valid_wafers:
    wafer_data = imp_long[imp_long['Wafer_Type'] == wafer]
    color = wafer_colors.get(wafer, 'gray')
    if len(wafer_data) > 0:
        fig.add_trace(go.Scatter(
            x=wafer_data['x_jittered'], y=wafer_data['Imp_Ohm'], mode='markers',
            marker=dict(color=color, size=3, opacity=0.8),
            name=wafer, showlegend=True,
            hovertemplate='<b>%{customdata[0]}</b><br>Imp: %{y:.0f}Ω<extra></extra>',
            customdata=wafer_data['Device_ID']
        ))

# Threshold line
fig.add_hline(1e6, line_dash="dash", line_color="red", line_width=3, annotation_text="1MΩ Threshold")
for i, (wafer, row) in enumerate(wafer_stats.iterrows()):
    stats_text = f"{wafer}: N={row['N_total']}, good={row['n_good']}, bad={row['n_bad']} ({row['pct_good']}%)"
    fig.add_annotation(
        x=i, y=1e12 * 0.8,  # Top-right of band (log scale)
        text=stats_text,
        showarrow=False,
        font=dict(size=11, color=wafer_colors.get(wafer, 'black')),
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='gray',
        borderwidth=1
    )

# Update layout for grouped scatter
x_labels = {pos: wafer for wafer, pos in wafer_positions.items()}
fig.update_layout(
    height=700,
    yaxis_title='Impedance (Ω)', yaxis_type='log',
    xaxis=dict(
        title='Wafer (jittered positions)',
        tickmode='array',
        tickvals=list(wafer_positions.values()),
        ticktext=list(wafer_positions.keys()),
        range=[-0.5, len(valid_wafers) - 0.5]
    ),
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': -0.3, 'xanchor': 'center', 'x': 0.5}, #
    title=f'All Wafers Impedance Scatter |Purple=Pt No Cactus/PoPt, Red=Rough Pt, Blue= PoPt with cactus, Green-PoPt No cactus'
)

fig.write_html('wafer_grouped_scatter.html')
fig.show(renderer='browser')
print("✅ Large grouped scatter plot saved as 'wafer_grouped_scatter.html'")
print("Legend shows wafers (colored), Good/Bad overlays. Hover for details!")

## End of SECTION 4 Grouped Scatter Plot by Wafer


Valid wafers: ['L1', 'R12', 'R3', 'R4', 'R6', 'R7', 'R8']
Number of L1 devices with any channel < 5kΩ: 2
  Device_ID  n_channels_below_5k
0        M2                    1
1        R5                   64
            N_total  n_good  n_bad  pct_good
Wafer_Type                                  
L1             1920    1659    261      86.4
R12             512     453     59      88.5
R3               64      56      8      87.5
R4               64      49     15      76.6
R6              384     302     82      78.6
R7              704     577    127      82.0
R8              640     536    104      83.8
✅ Large grouped scatter plot saved as 'wafer_grouped_scatter.html'
Legend shows wafers (colored), Good/Bad overlays. Hover for details!
